# Remove rooftop arrays by building intersection
NOTE: Requries BigPanelGEE.yml environment

**Use Google Earth Engine (GEE) to Remove Rooftop Arrays from Existing Solar Array Datasets**
* Inputs: *existingSolarArrayShapes.shp* local shape file and asset upload to GEE.
* Uses GEE to pull in [USA Structures](https://gee-community-catalog.org/projects/usa_structures/?h=ornl) dataset, calcualte the intersection with our newly-compiled solar array dataset, and remove arrays that are likely rooftop mounted. 
* Output: Ground mounted solar array dataset.

## Import Libraries

In [3]:
# Import libraries
import numpy as np
import pandas as pd
import geopandas as gpd
import os 
import ee
import geemap
from shapely.ops import unary_union

# Import gmseusUtils
import gmseusUtils as gu

## Initialize GEE

In [2]:
# Trigger the GEE authentication
ee.Authenticate()

# Initialize the cloud project
ee.Initialize(project='ee-stidjaco')


Successfully saved authorization token.


## Set paths and variables

In [ ]:
# Set folder paths
wd = r'S:\Users\stidjaco\R_files\BigPanel'
downloaded_path = os.path.join(wd, r'Data\Downloaded')
derived_path = os.path.join(wd, r'Data\Derived')
derivedTemp_path = os.path.join(derived_path, r'intermediateProducts')

# Set country name
countryName = 'USA'

# Set aoi shapefile path
aoi_path = os.path.join(wd, r'Data\Downloaded\CONUS_NoGreatLakes\CONUS_No_Great_Lakes.shp')

# Set initial solar array shapefile locations to compile
existingArraysPath = os.path.join(derivedTemp_path, r'existingDatasetArrayShapes.shp')
georectifiedArraysPath = os.path.join(derivedTemp_path, r'georectifiedSolarArrays.geojson')

# Set compiled solar array and panel shapefile locations (asset and local)
arraysLocalPath = os.path.join(derivedTemp_path, r'compiledArrayDataset.shp') # We create this file in this script
panelsLocalPath = os.path.join(derivedTemp_path, r'existingDatasetPanelShapes.shp') # This exists, and is created in script1
arraysAssetPath = r'projects/ee-stidjaco/assets/BigPanel/compiledArrayDataset'

# GM-SEUS initial output paths
gmseusArraysInitPath = os.path.join(derivedTemp_path, r'initialGMSEUS_Arrays.shp')
gmseusPanelsInitPath = os.path.join(derivedTemp_path, r'initialGMSEUS_Panels.shp')

# Get building database path
buildingAssetPath = r"projects/sat-io/open-datasets/VIDA_COMBINED/USA"

# Load the config from the text file
config = gu.load_config('config.txt')

# Set threshold for building area contained within array inferring rooftop
build_threshold = config['build_threshold'] # 50% of building area contained within array

# Other variables
gee_crs = config['gee_crs'] # native projection of Google Earth Engine exports
minPanelRowArea = config['minPanelRowArea'] # minimum area of panel to be considered
toCRS = config['to_crs']  # EPSG:6350 NAD83 (2011)

# Append toCRS with the EPSG prefix for use in GeoPandas
toCRS = f'EPSG:{toCRS}'

# Get US Boundary to subset global/non-CONUS datasets
aoi = gpd.read_file(aoi_path) # CONUS boundary shapefile
aoi = aoi.set_crs(epsg=4269) # Native projection of US boundary - NAD83
aoi = aoi.to_crs(toCRS) # Transform to projection of USPVDB
aoi['geometry'] = aoi.buffer(10) # Buffer US boundary by 10 meters to ensure that array bounds are not clipped

## STOP: Upload _arraysLocalPath_ to GEE Asset following the nomenclature of the arraysAssetPath prior to progressing

## Get solar array intersections with buildings and save as pandas df

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Google Global Building Asset Dataset

# Set asset path
buildingsAsset = ee.FeatureCollection(buildingAssetPath)

# Function to calculate the intersection and proportional area
def calculate_intersection_area(array):
    # Get the array geometry
    aoiTemp = array.geometry()

    # Filter the building vectors to the bounds of the aoi. 
    # For this Google Buildings Dataset, this always results in 29 features, only the last of which is the correct shape. We have not ascertained why this is the case. 
    # When we filter and acquire multipolygon geometries (if more than one building intersects), the correct shapes are index 29:numFeatures. 
    localBuildings = buildingsAsset.filterBounds(aoiTemp).geometry().geometries() 
    actualFootprintStart = 29 # First building footprint index for the Google Buildings Dataset. 
    numGeometries = localBuildings.size() # Get total number of polyigons
    validGeometries = localBuildings.slice(actualFootprintStart, numGeometries) # Slice the geometries from index 29 to the end
    localBuildings = ee.Geometry.MultiPolygon(validGeometries) # Convert the list of valid geometries back into a MultiPolygon

    # Now, acquire the intersecting area of roof and solar and get the rooftop proportion
    intersectionArea = aoiTemp.intersection(localBuildings, ee.ErrorMargin(1)).area(1)
    rooftopAreaProp = intersectionArea.divide(aoiTemp.area(1)).multiply(100).toInt() 

    # Set as new attribute and return the feature
    return array.set({'roofProp': rooftopAreaProp})

# Call asset arrays
#arraysAsset = geemap.shp_to_ee(arraysLocalPath) # Although this works, there are resulting memory issues if the asset is too large and not physcially uploaded to GEE first. So we'll use the asset path instead.
arraysAsset = ee.FeatureCollection(arraysAssetPath)

# Create a temporary folder within the derivedTemp_path to store the results
rooftopResultsPath = os.path.join(derivedTemp_path, 'rooftopResults')
if not os.path.exists(rooftopResultsPath):
    os.makedirs(rooftopResultsPath)

# Get assed ID list
assetIDList = arraysAsset.aggregate_array('tmpArrID').getInfo()

# Break the assetIDList into equal chunks smaller than 5000 to overcome GEE memory limitations. Then, for each chunk, get the corresponding feature collection and apply the calculate_intersection_area function to each feature. Append the results to the lists.
chunkSize = 1000 # This used to be 4999 to prevent memory issues using ORNL structures dataset, but with the google dataset, we needed multiple exports of csv to overcome memory issues.
chunks = [assetIDList[i:i + chunkSize] for i in range(0, len(assetIDList), chunkSize)]

# For each chunk, get the corresponding feature collection and apply the calculate_intersection_area function to each feature. Append the results to the lists.
for chunk in chunks:
    
    # Initialize lists
    id_list = []
    roofProp_list = []

    # Get the feature collection for the chunk
    arraysAssetChunk = arraysAsset.filter(ee.Filter.inList('tmpArrID', ee.List(chunk)))

    # Apply the function to each feature in the arrays collection, and append the results to the lists
    arrays_with_rooftop = arraysAssetChunk.map(calculate_intersection_area)
    for feature in arrays_with_rooftop.getInfo()['features']:
        id_list.append(feature['properties']['tmpArrID'])
        roofProp_list.append(feature['properties']['roofProp'])

    # Create a dictionary and convert to a dataframe
    data = {'tmpArrID': id_list, 'roofProp': roofProp_list}
    arraysRooftopDf = pd.DataFrame(data)

    # Export the result to CSV
    arraysRooftopDf.to_csv(os.path.join(rooftopResultsPath, 'arraysRooftopProp'+str(chunk[0])+'.csv'), index=False)

# Call in all the csv files and concatenate them into one dataframe from the rooftopResultsPath
arraysRooftopDf = pd.concat([pd.read_csv(os.path.join(rooftopResultsPath, f)) for f in os.listdir(rooftopResultsPath)], ignore_index=True)

# Check to ensure indexing logic is correct
print("Number of arrays assessed: ", len(arraysRooftopDf))
print("Correct total number of arrays: ", arraysAsset.size().getInfo())

# Export dataframe to derivedTemp_path
arraysRooftopDf.to_csv(os.path.join(derivedTemp_path, 'arraysRooftopProp.csv'), index=False)

Number of arrays assessed:  16691
Correct total number of arrays:  16691


## Remove buildings from existing and digitized solar array and panel databases

### Arrays

In [ ]:
# Call local arrays
arraysLocal = gpd.read_file(arraysLocalPath)

# Print length of local arrays
print("Original number of arrays: ", len(arraysLocal))

# Call the CSV
arraysRooftopDf = pd.read_csv(os.path.join(derivedTemp_path, 'arraysRooftopProp.csv'))

# Merge the dataframes on a common identifier
mergedArrays = arraysLocal.merge(arraysRooftopDf[['tmpArrID', 'roofProp']], on='tmpArrID')

# Drop mergedArrays with a rooftopProp below the threshold
mergedArrays = mergedArrays[mergedArrays['roofProp'] <= build_threshold]

# Print length of merged arrays
print("Number of arrays after filtering: ", len(mergedArrays))

# Print total area of ground-mounted arrays in square km 
print("Total area of ground-mounted arrays: ", mergedArrays['area'].sum() / 10**6)

# Print the number of rooftop arrays removed 
print("Number of rooftop arrays removed: ", len(arraysLocal) - len(mergedArrays))

# Drop the temporary ID column
mergedArrays = mergedArrays.drop(columns=['tmpArrID'])

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Set a subset identifier for GEE memory limitations

# Create a 'subset' column for processing panels in GEE. 
# This is a unique identifier for each array depending on its size. It should be numbers 0 through 999. 
# Larger arrays requrie more vectorization and more GEE memory, and should be procesed in smaller batches.
# Calculate the 90th, 50th, and 10th, percentile area values
percentile_90 = mergedArrays['area'].quantile(0.90)
percentile_50 = mergedArrays['area'].quantile(0.50)
percentile_10 = mergedArrays['area'].quantile(0.10)

# Set subset values based on the 90th percentile area value
percentile_90thSubset = 500 # Number of subset values for arrays larger than the 90th percentile area
percentile_50thSubset = 400 # Number of subset values for arrays between the 50th and 90th percentile area
percentile_10thSubset = 99 # Number of subset values for arrays smaller than the 10th percentile area

# Function to assign subset values
def assign_subset(area, percentile_90):
    if area > percentile_90:
        return np.random.randint(0, percentile_90thSubset)
    elif area <= percentile_90 and area > percentile_50:
        return np.random.randint(percentile_90thSubset, percentile_90thSubset + percentile_50thSubset)
    else:
        return np.random.randint(percentile_90thSubset + percentile_50thSubset, percentile_90thSubset + percentile_50thSubset + percentile_10thSubset)

# Apply the function to create the 'subset' column
mergedArrays['subset'] = mergedArrays['area'].apply(assign_subset, percentile_90=percentile_90)

# Print min max subset values
print(f'Minimum subset value: {mergedArrays["subset"].min()}')
print(f'Maximum subset value: {mergedArrays["subset"].max()}')

# Set a initID column that is the row index
mergedArrays = mergedArrays.reset_index(drop=True)
mergedArrays['arrayID'] = mergedArrays.index

# Export the merged arrays to shapefile
mergedArrays.to_file(gmseusArraysInitPath)

Original number of arrays:  16691
Number of arrays after filtering:  14905
Total area of ground-mounted arrays:  3055.8286719458915
Number of rooftop arrays removed:  1786
Minimum subset value: 0
Maximum subset value: 998


### Panels

In [10]:
# Re-call GM-SEUS initial arrays and call the panel-row data (both are already in the correct projection)
gmseusArrays = gpd.read_file(gmseusArraysInitPath)
panelsLocal = gpd.read_file(panelsLocalPath)

# Print the original number of panels
print("Original number of panels: ", len(panelsLocal))

# Spatially join gmseus arrays to panels, copy the arrayID to the panels, and drop the index columns. 
panelsLocal = gpd.sjoin(panelsLocal, gmseusArrays[['arrayID', 'geometry']], how='left', predicate='intersects')
panelsLocal = panelsLocal.reset_index(drop=True)
panelsLocal = panelsLocal.drop(columns=['index_left', 'index_right'], errors='ignore')

# Drop panels that do not have an arrayID
gmPanelsLocal = panelsLocal.dropna(subset=['arrayID'])

# Print the number of panels after filtering
print("Number of panels after filtering: ", len(gmPanelsLocal))

# Print the number of rooftop panels removed
print("Number of rooftop panels removed: ", len(panelsLocal) - len(gmPanelsLocal))

# Print the total area of ground-mounted panels in square km
print("Total area of ground-mounted panels: ", gmPanelsLocal['area'].sum() / 10**6)

# Print the number of unique initIDs in the panels
print("Number of GMSEUS arrays with existing panels: ", len(gmPanelsLocal['arrayID'].unique()))

# Drop arrayID column
gmPanelsLocal = gmPanelsLocal.drop(columns=['arrayID'])

# Save a new panelID column that is the row index (after resetting the index)
gmPanelsLocal = gmPanelsLocal.reset_index(drop=True)
gmPanelsLocal['panelID'] = gmPanelsLocal.index

# Export the panels to shapefile
gmPanelsLocal.to_file(gmseusPanelsInitPath)

Original number of panels:  1076800
Number of panels after filtering:  1071181
Number of rooftop panels removed:  5714
Total area of ground-mounted panels:  137.31026341708852
Number of GMSEUS arrays with existing panels:  4470
